多头注意力机制

In [2]:
# 导入相关需要的包
import math
import torch
import torch.nn as nn

import warnings
warnings.filterwarnings(action="ignore") # 忽略警告

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self,hidden_dim,head_num):
        super().__init__()

        self.hidden_dim = hidden_dim
        self.head_num = head_num
        self.head_dim = hidden_dim // head_num

        self.q = nn.Linear(hidden_dim,hidden_dim)
        self.k = nn.Linear(hidden_dim,hidden_dim)
        self.v = nn.Linear(hidden_dim,hidden_dim)
        self.output_proj = nn.Linear(hidden_dim,hidden_dim)

        self.dropout = nn.Dropout(0.1)
    
    def forward(self,x,attention_mask = None):
        batch_size,seq_len,_ = x.size()

        # 1.分别计算q、k、v
        q = self.q(x)
        k = self.k(x)
        v = self.v(x)
        print(q.shape) # [batch_size, seq_len, hidden_dim]

        # 2.shape 变成 （batch_size, num_head, seq_len, head_dim）
        # [batch_size, seq_len, hidden_dim] -> [batch_size, seq_len, num_head, head_dim] -> [batch_size, num_head, seq_len, head_dim]
        q_state = q.view(batch_size,seq_len,self.head_num,self.head_dim).transpose(1,2)
        k_state = k.view(batch_size,seq_len,self.head_num,self.head_dim).transpose(1,2)
        v_state = v.view(batch_size,seq_len,self.head_num,self.head_dim).transpose(1,2)
        print(q_state.shape)

        # 3.计算attention_weight
        attention_value = q_state @ k_state.transpose(-1,-2)
        attention_weight = attention_value / math.sqrt(self.hidden_dim)
        # [batch_size, num_head, seq_len, seq_len]
        print(attention_mask.shape)
        if attention_mask is not None:
            attention_weight = attention_weight.masked_fill(
                attention_mask == 0,float("-inf")
            )
        
        # 4.算softmax
        attention_weight = torch.softmax(attention_weight,dim=-1)
        print(attention_weight.shape)

        #5.dropout
        attention_weight = self.dropout(attention_weight)

        # 6.计算output
        output_mid = attention_weight @ v_state
        print(output_mid.shape)
        # 重新变成 (batch, seq_len, num_head, head_dim)
        # 这里的 contiguous() 是相当于返回一个连续内存的 tensor，一般用了 permute/tranpose 都要这么操作
        # 如果后面用 Reshape 就可以不用这个 contiguous()，因为 view 只能在连续内存中操作
        # view和transpose都只是改变了索引规则，不是真的改变了其存储形式
        output_mid = output_mid.transpose(1, 2).contiguous()
        print(output_mid.shape)
        # 变成 (batch, seq, hidden_dim),
        output = output_mid.view(batch_size, seq_len, -1)
        print(output.shape)
        return self.output_proj(output)


In [23]:
attention_mask = (
    torch.tensor(
        [
            [0, 1],
            [0, 0],
            [1, 0],
        ]
    )
    .unsqueeze(1)
    .unsqueeze(2)
    .expand(3, 8, 2, 2)
)
# print(attention_mask.shape)
# print(attention_mask)

x = torch.rand(3, 2, 128)
net = MultiHeadAttention(128, 8)
net(x, attention_mask).shape

torch.Size([3, 2, 128])
torch.Size([3, 8, 2, 16])
torch.Size([3, 8, 2, 2])
torch.Size([3, 8, 2, 2])
torch.Size([3, 8, 2, 16])
torch.Size([3, 2, 8, 16])
torch.Size([3, 2, 128])


torch.Size([3, 2, 128])